# 05 · Catherine Eddowes · Mitre Square — prueba visual con evidencia bloqueada

**Archivo original:** `web/assets/closeups/eddowes.jpg` (1280×720). La imagen existente ya es fotorrealista; cambios fuertes pueden empeorarla o destruir pistas esenciales.

**Invariantes de gameplay:** cara (253,306), cuello (360,337), vientre (766,398), delantal (1110,536), muñeca (903,100). Se utilizan exactamente las coordenadas y radios declarados en el motor. Esas áreas están **excluidas** de la máscara de edición. Este notebook trabaja una región de fondo, no rostros, objetos forenses ni heridas.

**Flujo recomendado:** primero A/B de gradación local sin GPU; si hay una razón visual concreta, habilitar `RUN_AI=True` para un inpainting muy limitado con modelo abierto sobre T4. No ejecutar los tres notebooks de escenas al mismo tiempo.

La versión resultante **no se integra ni publica automáticamente**. Comprobar pistas, legibilidad, estética y FPS en Android.

In [ ]:
%pip -q install 'diffusers>=0.35,<0.40' 'transformers>=4.48,<5' 'accelerate>=1.0' 'safetensors>=0.5' 'Pillow>=10' 'numpy>=1.24' 'requests>=2.31'

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance, ImageOps, ImageChops
from IPython.display import display
import base64, io, json, math, hashlib, random, requests, shutil
REPO = "gabsvm/HorrorPlay"
COMMIT = "0681cfee36921ec2bd372f8ae8080a401f8bd63b"

def source(path):
    url = f"https://api.github.com/repos/{REPO}/contents/{path}"
    r = requests.get(url, params={"ref": COMMIT}, headers={"Accept":"application/vnd.github+json"}, timeout=90)
    r.raise_for_status()
    data = r.json()
    if data.get("encoding") != "base64":
        raise RuntimeError(f"GitHub no devolvió el JPG/PNG en base64: {path}")
    b = base64.b64decode(data["content"])
    result = Image.open(io.BytesIO(b)).convert("RGB" if path.lower().endswith((".jpg", ".jpeg")) else "RGBA")
    return result, hashlib.sha256(b).hexdigest()

def font(size=20, bold=False):
    files = ["/usr/share/fonts/truetype/dejavu/DejaVuSerif-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf",
             "/usr/share/fonts/truetype/liberation2/LiberationSerif-Regular.ttf"]
    for f in files:
        if Path(f).exists(): return ImageFont.truetype(f, size)
    return ImageFont.load_default()

def compare(images, thumb=(640,360), labels=None):
    w,h=thumb
    sheet=Image.new('RGB',(w*len(images),h+36),(17,18,20))
    dr=ImageDraw.Draw(sheet)
    for i,im in enumerate(images):
        sheet.paste(im.convert('RGB').resize((w,h), Image.Resampling.LANCZOS),(i*w,36))
        dr.text((i*w+12,10), labels[i] if labels else str(i),fill=(245,241,229),font=font(15))
    return sheet

def finish(folder, manifest, outputs):
    manifest.update({"repo":REPO,"source_commit":COMMIT,"status":"CANDIDATO - No integrado ni testeado en Android"})
    (folder/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
    archive=shutil.make_archive(str(folder),'zip',folder)
    print('Exportado:', archive, 'archivos:', len(list(folder.iterdir())))
    from google.colab import files
    files.download(archive)

In [ ]:
out=Path('/content/horrorplay_eddowes_candidates');out.mkdir(parents=True,exist_ok=True)
ORIGINAL, SHA=source('web/assets/closeups/eddowes.jpg')
assert ORIGINAL.size==(1280,720)
print('Fuente sha256:',SHA)
display(ORIGINAL.resize((960,540)))
# Puntos interactivos REALES de la escena, no coordenadas estimadas.
HOTSPOTS=[(253, 306, 60), (360, 337, 45), (766, 398, 85), (1110, 536, 70), (903, 100, 55)]
print('Preservar estos hotspots:','cara (253,306), cuello (360,337), vientre (766,398), delantal (1110,536), muñeca (903,100)')
REGION=[(1120, 42), (1260, 42), (1260, 286), (1115, 286)]
ROI=(768, 0, 1280, 512)

mask=Image.new('L',ORIGINAL.size,0);dr=ImageDraw.Draw(mask)
dr.polygon(REGION,fill=255)
# Zonas de seguridad: no editar nunca dentro del radio de cada pista + 24 px.
for x,y,r in HOTSPOTS:dr.ellipse((x-r-24,y-r-24,x+r+24,y+r+24),fill=0)
red=Image.new('RGBA',ORIGINAL.size,(215,35,35,0));red.putalpha(mask.point(lambda x:round(x*.55)))
display(Image.alpha_composite(ORIGINAL.convert('RGBA'),red).resize((960,540)))
assert mask.getbbox(), 'La región de edición está vacía'
for x,y,r in HOTSPOTS:
    assert mask.getpixel((x,y))==0, 'Hotspot dentro de la máscara'

## Variaciones seguras de contraste (no usa GPU)

In [ ]:
# Primer paso SIN GPU: corrección mínima de materiales / contraste, solo dentro de la máscara.
preview=[]
for strength in (0.0, .25, .45):
    # 0% = original; 25/45% = mezcla con una mejora de contraste y nitidez local,
    # deliberadamente conservadora para no introducir falsas pistas.
    corrected=ImageEnhance.Contrast(ORIGINAL).enhance(1.12)
    corrected=ImageEnhance.Sharpness(corrected).enhance(1.20)
    partial=Image.blend(ORIGINAL,corrected,strength)
    out_im=Image.composite(partial,ORIGINAL,mask.filter(ImageFilter.GaussianBlur(5)))
    preview.append(out_im)
    if strength>0:out_im.save(out/f'graded_{int(strength*100)}.jpg',quality=88,optimize=True)
compare(preview,(640,360),['Original','Gradación 25%','Gradación 45%']).save(out/'AB_grading_640.png')
display(compare(preview,(640,360),['Original','25%','45%']))

## IA de inpainting opcional (T4)

Solo para una textura de fondo que realmente necesite redibujarse. Usa los mismos pesos abiertos probados en Thomas/Nat, sin una API externa de pago. En el primer uso puede descargar varios gigabytes; revisa unidades de Colab. Si editas la máscara revisa primero el rojo y verifica que todos los puntos de pista queden negros.

In [ ]:
# Opcional. Cambia a True solo si hace falta textura nueva, no para modificar pruebas.
RUN_AI=False
SEEDS=[1888]
if RUN_AI:
    import torch
    from diffusers import AutoPipelineForInpainting
    assert torch.cuda.is_available(),'Activa GPU T4'
    MODEL_ID='stable-diffusion-v1-5/stable-diffusion-inpainting'
    pipe=AutoPipelineForInpainting.from_pretrained(MODEL_ID,torch_dtype=torch.float16,
                                    use_safetensors=False,variant='fp16')
    pipe.enable_model_cpu_offload(); pipe.enable_vae_slicing()
    generated=[]
    for seed in SEEDS:
        local_mask=mask.crop(ROI)
        assert local_mask.getbbox()
        input_crop=ORIGINAL.crop(ROI)
        ai=pipe(prompt='dark wet Victorian street paving stones in deserted London square at night, realistic period stone textures, subtle aged pavement detail, photographic cinematic ground only',
                negative_prompt='person, face, hands, new objects, blood, medical details, symbols, lettering, text, cartoon, CGI, distorted geometry',
                image=input_crop,mask_image=local_mask,
                width=512,height=512,num_inference_steps=26,
                guidance_scale=6,strength=.35,
                generator=torch.Generator(device='cpu').manual_seed(seed)).images[0].convert('RGB')
        soft=local_mask.filter(ImageFilter.GaussianBlur(6))
        candidate=ORIGINAL.copy()
        candidate.paste(Image.composite(ai,input_crop,soft),ROI)
        candidate.save(out/f'eddowes_ai_seed{seed}.png')
        generated.append(candidate)
    comparison=compare([ORIGINAL,*generated],(640,360),['Original']+[f'AI seed {s}' for s in SEEDS])
    display(comparison); comparison.save(out/'AB_AI_640.png')
else:
    print('Inpainting desactivado: ediciones seguras sin gastar GPU. Para usar IA, establece RUN_AI=True y reejecuta.')

In [ ]:
finish(out,{'sha256':SHA,'asset':'eddowes.jpg', 'hotspots':HOTSPOTS,
    'region':REGION,'ROI':ROI,'ai_enabled':RUN_AI, 'seeds':SEEDS if RUN_AI else [],
    'review':'Comparar originales en escala 640x360; confirmar pistas y ausencia de nuevos objetos'},[])